In [1]:
import pandas as pd

In [4]:
df=pd.read_csv('day5_data.csv')

In [5]:
df

,Customer_ID,City,Education_Level,ZipCode,House_Price,Target_Label
0,1,New York,High School,ZIP_100,250000,Churn
1,2,London,Bachelors,ZIP_101,320000,Stay
2,3,Paris,Masters,ZIP_102,450000,Stay
3,4,New York,PhD,ZIP_100,500000,Stay
4,5,London,High School,ZIP_103,210000,Churn
5,6,Paris,Bachelors,ZIP_102,340000,Churn
6,7,New York,Masters,ZIP_104,410000,Stay
7,8,London,PhD,ZIP_101,550000,Stay
8,9,Paris,High School,ZIP_104,230000,Churn
9,10,New York,Bachelors,ZIP_100,290000,Churn


In [6]:
df.columns

Index(['Customer_ID', 'City', 'Education_Level', 'ZipCode', 'House_Price',
       'Target_Label'],
      dtype='str')

In [7]:
x=df.drop(['House_Price','Target_Label'],axis=1)

In [8]:
y_cat=df['Target_Label']

In [9]:
y_num=df['House_Price']

In [23]:
y_train_num=y_num.loc[x_train.index]

In [25]:
y_test_num=y_num.loc[x_test.index]

In [11]:
from sklearn.model_selection import train_test_split as tts

In [12]:
x_train,x_test,y_train,y_test=tts(x,y_cat,test_size=0.2,random_state=42)

In [14]:
print('---Original training data---')
print(x_train)

---Original training data---
    Customer_ID      City Education_Level  ZipCode
8             9     Paris     High School  ZIP_104
5             6     Paris       Bachelors  ZIP_102
11           12     Paris             PhD  ZIP_102
3             4  New York             PhD  ZIP_100
18           19  New York         Masters  ZIP_101
16           17    London     High School  ZIP_102
13           14    London       Bachelors  ZIP_101
2             3     Paris         Masters  ZIP_102
9            10  New York       Bachelors  ZIP_100
19           20    London             PhD  ZIP_103
4             5    London     High School  ZIP_103
12           13  New York     High School  ZIP_104
7             8    London             PhD  ZIP_101
10           11    London         Masters  ZIP_103
14           15     Paris         Masters  ZIP_100
6             7  New York         Masters  ZIP_104


In [15]:
#Label Encoding: use to label the categorical data in Y. 

In [16]:
print('*****Label Encoder*****')

*****Label Encoder*****


In [91]:
from sklearn.preprocessing import LabelEncoder

In [92]:
y_train

8     Churn
5     Churn
11     Stay
3      Stay
18     Stay
16    Churn
13     Stay
2      Stay
9     Churn
19     Stay
4     Churn
12    Churn
7      Stay
10     Stay
14    Churn
6      Stay
Name: Target_Label, dtype: str

In [93]:
y_test

0     Churn
17     Stay
15     Stay
1      Stay
Name: Target_Label, dtype: str

In [94]:
LE=LabelEncoder()

In [96]:
y_train_LE=LE.fit_transform(y_train)

In [97]:
y_test_LE=LE.transform(y_test)

In [31]:
y_train_LE

array([0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1])

In [98]:
y_train_LE=pd.Series(
    y_train_LE,
    name="Target_Label",
    index=y_train.index
)

In [106]:
y_train_L=y_train_LE.to_frame()

In [107]:
y_train_L

,Target_Label
8,0
5,0
11,1
3,1
18,1
16,0
13,1
2,1
9,0
19,1


In [32]:
y_test_LE

array([0, 1, 1, 1])

In [38]:
df.head()

,Customer_ID,City,Education_Level,ZipCode,House_Price,Target_Label
0,1,New York,High School,ZIP_100,250000,Churn
1,2,London,Bachelors,ZIP_101,320000,Stay
2,3,Paris,Masters,ZIP_102,450000,Stay
3,4,New York,PhD,ZIP_100,500000,Stay
4,5,London,High School,ZIP_103,210000,Churn


In [33]:
#Using column transform to treat all columns of X in one go 

In [81]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder
from category_encoders import LeaveOneOutEncoder
from sklearn.model_selection import KFold

In [53]:
cv_splitter = KFold(n_splits=3, shuffle=True, random_state=42)

In [75]:
preprocessor=ColumnTransformer(
    transformers=[
        ('education_ordinal',OrdinalEncoder(categories=[['High School','Bachelors','Masters','PhD']],dtype=int),['Education_Level']),
        ('city_ohe',OneHotEncoder(drop='first',sparse_output=False,dtype=int),['City']),
        ('zip_k_fold', TargetEncoder(cv=cv_splitter),['ZipCode']),
        ('drop_col','drop',['Customer_ID'])
    ],
    remainder='passthrough'
).set_output(transform='pandas')         

In [76]:
x_train_transformed=preprocessor.fit_transform(x_train,y_train)

In [77]:
x_test_transformed=preprocessor.transform(x_test)

In [78]:
x_train_transformed

,education_ordinal__Education_Level,city_ohe__City_New York,city_ohe__City_Paris,zip_k_fold__ZipCode
8,0,0,1,1.000000
5,1,0,1,1.000000
11,3,0,1,0.361217
3,3,1,0,0.000000
18,2,1,0,1.000000
16,0,0,0,1.000000
13,1,0,0,1.000000
2,2,0,1,0.361217
9,1,1,0,0.484765
19,3,0,0,0.484765


In [79]:
x_test_transformed

,education_ordinal__Education_Level,city_ohe__City_New York,city_ohe__City_Paris,zip_k_fold__ZipCode
0,0,1,0,0.386353
17,1,0,1,0.386353
15,3,1,0,0.642567
1,1,0,0,1.000000


In [80]:
#Testing leave one out for zipcode

In [82]:
loo_encoder = LeaveOneOutEncoder(cols=['ZipCode'])

In [84]:
x_train['Zip_LOO_Enc'] = loo_encoder.fit_transform(x_train[['ZipCode']], y_train)
x_test['Zip_LOO_Enc'] = loo_encoder.transform(x_test[['ZipCode']])

In [85]:
x_train

,Customer_ID,City,Education_Level,ZipCode,Zip_LOO_Enc
8,9,Paris,High School,ZIP_104,0.500000
5,6,Paris,Bachelors,ZIP_102,0.666667
11,12,Paris,PhD,ZIP_102,0.333333
3,4,New York,PhD,ZIP_100,0.000000
18,19,New York,Masters,ZIP_101,1.000000
16,17,London,High School,ZIP_102,0.666667
13,14,London,Bachelors,ZIP_101,1.000000
2,3,Paris,Masters,ZIP_102,0.333333
9,10,New York,Bachelors,ZIP_100,0.500000
19,20,London,PhD,ZIP_103,0.500000


In [86]:
#You can use any one of them, k-fold or leave-one-out encoder 